In [ ]:
import nemo.collections.asr as nemo_asr

NEMO_PATH = "../../data/models/titanet_large/speakerverification_en_titanet_large.nemo"
model = nemo_asr.models.EncDecSpeakerLabelModel.restore_from(NEMO_PATH)

# emb = model.get_embedding("../../data/audio/audio-test.wav")
# print(type(emb), getattr(emb, "shape", None))

In [ ]:
import torch
import numpy as np

def compute_embeddings_nemo_tensor_try(
    model,
    audio_int16: np.ndarray,
    sr: int,
    windows,
    slice_waveform_int16,
    window_sec: float,
    batch_size: int = 128,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()

    T = int(round(window_sec * sr))  # фикс длина, раз окна fixed-length

    all_embs = []
    all_meta = []

    def iter_batches(items, bs):
        for i in range(0, len(items), bs):
            yield items[i:i+bs]

    with torch.no_grad():
        for batch_windows in iter_batches(windows, batch_size):
            # (B, T)
            batch = np.zeros((len(batch_windows), T), dtype=np.float32)

            for j, (t0, t1) in enumerate(batch_windows):
                x = slice_waveform_int16(audio_int16, sr, t0, t1).astype(np.float32) / 32768.0
                if len(x) != T:
                    # если у тебя окна строго fixed-length, этого быть не должно
                    # но на всякий случай — обрежем/паддим
                    x = x[:T]
                    if len(x) < T:
                        x = np.pad(x, (0, T - len(x)))
                batch[j] = x

            sig = torch.from_numpy(batch).to(device)
            sig_len = torch.full((sig.shape[0],), T, device=device, dtype=torch.long)

            # попытка "тензорного" вызова
            out = model(input_signal=sig, input_signal_length=sig_len)

            # дальше нужно понять структуру out:
            # часто это logits, а embedding можно достать из decoder/emb_layers
            # поэтому сначала просто посмотрим тип/shape:
            return out  # диагностически

# диагностика (запусти и покажи, что вернёт)
out = compute_embeddings_nemo_tensor_try(
    model=model,
    audio_int16=audio,
    sr=sr,
    windows=windows[:2],
    slice_waveform_int16=slice_waveform_int16,
    window_sec=2.0,
    batch_size=2,
)
print(type(out))